# 04 — Tool calling: o LLM pede, o Python calcula

No notebook 03 o RAG errou "quanto gastei em 2025?". A solução é não pedir ao modelo
que faça contas: damos a ele **ferramentas** (funções Python) e ele decide qual chamar.

O fluxo:

1. enviamos a pergunta **e a descrição das ferramentas**;
2. o modelo responde com um *pedido*: "chame `total_gasto(inicio='2025-01-01', fim='2025-12-31')`";
3. **nós** executamos a função (com pandas, conta exata) e devolvemos o resultado;
4. o modelo redige a resposta final com esse número.

O modelo nunca executa código — ele só escolhe a função e os argumentos.

In [ ]:
import json
from pathlib import Path

import ollama
from ollama._utils import convert_function_to_tool

from faturas.dados import carregar_faturas
from faturas.llm import MODELO_CHAT
from faturas.rag import carregar_ou_criar_indice, responder
from faturas.tools import _instrucoes, criar_ferramentas, perguntar_com_tools

df = carregar_faturas("../data")
ferramentas = criar_ferramentas(df)
por_nome = {f.__name__: f for f in ferramentas}

## O que o modelo enxerga

O cliente `ollama` transforma cada função num **schema JSON**, a partir do nome,
dos type hints e da docstring. É só isso que o modelo vê — ele não lê o código.

In [ ]:
schema = convert_function_to_tool(por_nome["total_gasto"])
print(json.dumps(schema.model_dump(exclude_none=True), indent=2, ensure_ascii=False))

## O loop, passo a passo

Vamos fazer manualmente o que `perguntar_com_tools` faz, imprimindo cada etapa.

In [ ]:
mensagens = [
    {"role": "system", "content": _instrucoes()},
    {"role": "user", "content": "Quanto gastei em 2025?"},
]
resposta = ollama.chat(model=MODELO_CHAT, messages=mensagens, tools=ferramentas,
                       think=False, options={"temperature": 0})
print("conteúdo de texto:", repr(resposta.message.content))
for chamada in resposta.message.tool_calls or []:
    print("🔧 o modelo pediu:", chamada.function.name, dict(chamada.function.arguments))

O modelo **não respondeu** — ele pediu uma ferramenta. Agora executamos e devolvemos
o resultado numa mensagem com papel `tool`:

In [ ]:
mensagens.append(resposta.message)
for chamada in resposta.message.tool_calls or []:
    resultado = por_nome[chamada.function.name](**chamada.function.arguments)
    print("🐍 Python calculou:", resultado)
    mensagens.append({"role": "tool", "tool_name": chamada.function.name, "content": json.dumps(resultado)})

final = ollama.chat(model=MODELO_CHAT, messages=mensagens, tools=ferramentas,
                    think=False, options={"temperature": 0})
print("💬 resposta final:", final.message.content)

## Usando a função pronta

In [ ]:
for pergunta in [
    "Quanto gastei em 2025?",
    "Qual mês eu gastei mais?",
    "Quanto gastei no Pizzaria Exemplo?",
    "Quanto gastei no primeiro semestre de 2026?",
]:
    print(f"❓ {pergunta}\n💬 {perguntar_com_tools(pergunta, df)}\n")

## A docstring é um prompt

Na primeira versão deste projeto, a docstring de `total_gasto` dizia só
*"Soma dos gastos entre duas datas"*. Para "quanto gastei em 2025?", o modelo preferia
chamar `gastos_por_mes` e **somar os 12 meses de cabeça** — errando o total.

A correção não foi no código, foi no **texto**: dizer quando usar cada ferramenta
("Use para 'quanto gastei em um ano'", "Não some estes valores: use total_gasto")
e fixar `temperature=0` para o modelo fazer sempre a mesma escolha.

In [ ]:
print(por_nome["total_gasto"].__doc__)
print(por_nome["gastos_por_mes"].__doc__)

## RAG × Tools × valor real

In [ ]:
indice = carregar_ou_criar_indice(df, Path("../data/embeddings.npz"))
pergunta = "Quanto gastei no total em 2025?"
real = por_nome["total_gasto"]("2025-01-01", "2025-12-31")

print(f"📚 RAG:   {responder(pergunta, indice)[-200:]}\n")
print(f"🔧 Tools: {perguntar_com_tools(pergunta, df)}\n")
print(f"🐼 Real:  R$ {real:,.2f}".replace(",", "_").replace(".", ",").replace("_", "."))

## Quando usar cada abordagem

| Pergunta | Melhor abordagem | Por quê |
|---|---|---|
| "Quando comprei no X?", "Tenho gasto com pet?" | **RAG** | encontrar poucos registros por significado |
| "Quanto gastei em 2025?", "Qual mês gastei mais?" | **Tools** | conta exata sobre muitos registros |
| "Explique o que é IOF" | **LLM puro** | conhecimento geral, não depende dos seus dados |

Sistemas reais costumam combinar os três: o modelo decide se busca, se calcula ou se responde direto.
Um bom próximo exercício é transformar a busca do RAG numa **ferramenta** (`buscar_por_significado`)
e deixar o modelo escolher entre ela e as outras.